# Edge Attribution Patching (EAP) Circuit Discovery on llama3-0.5B

## 1. Installation

In [1]:
!pip install torch transformers datasets accelerate tqdm networkx matplotlib seaborn pandas numpy --quiet

In [6]:
from huggingface_hub import login
login(new_session=False)

## 2. Imports and Setup

In [2]:
import os
import json
import random
import pickle
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Tuple, Optional, Union, Any, Set
from collections import defaultdict
from pathlib import Path
from enum import Enum
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from datasets import load_dataset

warnings.filterwarnings('ignore')

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.9.0+cu126, CUDA: True
GPU: NVIDIA A100-SXM4-80GB


## 3. Experiment Configuration

In [3]:
@dataclass
class ExperimentConfig:
    # Model
    model_name: str = "meta-llama/Llama-3.2-1B"
    discovery_algorithms: List[str] = field(default_factory=lambda: ["top_k"])

    # Hardware
    num_gpus: int = 1
    batch_size: int = 64
    use_fp16: bool = True

    # Dataset
    pile10k_samples: int = -1  # -1 = all
    c4_samples: int = 10000
    max_seq_length: int = 128

    # Edge scoring
    edge_percentile: float = 0.1
    layer_gap_max: int = 5  # L -> L+5 max
    use_absolute_values: bool = True

    # GLUE
    glue_samples_per_task: int = 200
    glue_tasks: List[str] = field(default_factory=lambda: [
        "cola", "sst2", "mrpc", "qqp", "stsb", "mnli", "qnli", "rte", "wnli"
    ])

    # Other
    seed: int = 42
    output_dir: str = "./circuit_outputs"
    save_intermediate: bool = True
    scoring_methods: List[str] = field(default_factory=lambda: ["standard_Relp"])

    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)

    def save(self, path: str):
        with open(path, 'w') as f:
            json.dump(asdict(self), f, indent=2)

config = ExperimentConfig()
set_seed(config.seed)

print("Config initialized")
for k, v in asdict(config).items():
    print(f"  {k}: {v}")

Config initialized
  model_name: meta-llama/Llama-3.2-1B
  discovery_algorithms: ['top_k']
  num_gpus: 1
  batch_size: 64
  use_fp16: True
  pile10k_samples: -1
  c4_samples: 10000
  max_seq_length: 128
  edge_percentile: 0.1
  layer_gap_max: 5
  use_absolute_values: True
  glue_samples_per_task: 200
  glue_tasks: ['cola', 'sst2', 'mrpc', 'qqp', 'stsb', 'mnli', 'qnli', 'rte', 'wnli']
  seed: 42
  output_dir: ./circuit_outputs
  save_intermediate: True
  scoring_methods: ['standard_Relp']


## 4. Node and Edge Definitions

In [4]:
class NodeType(Enum):
    EMBED = "embed"
    ATTN_HEAD = "attn_head"
    MLP = "mlp"
    OUTPUT = "output"

@dataclass(frozen=True)
class Node:
    node_type: NodeType
    layer: int
    head: Optional[int] = None

    def __str__(self):
        if self.node_type == NodeType.EMBED:
            return "Embed"
        elif self.node_type == NodeType.OUTPUT:
            return "Output"
        elif self.node_type == NodeType.MLP:
            return f"MLP_{self.layer}"
        elif self.node_type == NodeType.ATTN_HEAD:
            return f"Attn_{self.layer}.{self.head}"
        return f"{self.node_type.value}_{self.layer}"

    def __hash__(self):
        return hash((self.node_type, self.layer, self.head))

@dataclass
class Edge:
    source: Node
    target: Node
    weight: float = 0.0

    def __hash__(self):
        return hash((self.source, self.target))

    def __eq__(self, other):
        return (
            isinstance(other, Edge) and
            self.source == other.source and
            self.target == other.target
        )

    def __str__(self):
        return f"{self.source} -> {self.target} (w={self.weight:.6f})"

## 5. Model Wrapper

In [7]:
# ============ CELL 5: Model Wrapper  ============
class ModelWrapper:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading model: {config.model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            torch_dtype=torch.float16 if config.use_fp16 else torch.float32,
            low_cpu_mem_usage=True
        ).to(self.device)
        self.model.eval()

        # Extract model dimensions
        cfg = self.model.config
        self.n_layers = cfg.num_hidden_layers
        self.n_heads = cfg.num_attention_heads
        self.d_model = cfg.hidden_size
        self.d_head = self.d_model // self.n_heads

        # Build computation graph
        self.nodes, self.edges = self._build_graph()

        # Hook storage
        self.activations = {}
        self.gradients = {}
        self.hooks = []

        print(f"Model loaded: {self.n_layers} layers, {self.n_heads} heads, d_model={self.d_model}")
        print(f"Graph: {len(self.nodes)} nodes, {len(self.edges)} edges")
        if torch.cuda.is_available():
            print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    def _build_graph(self):
        """Build computation graph of nodes and edges."""
        nodes = [Node(NodeType.EMBED, -1)]

        for layer in range(self.n_layers):
            for head in range(self.n_heads):
                nodes.append(Node(NodeType.ATTN_HEAD, layer, head))
            nodes.append(Node(NodeType.MLP, layer))

        nodes.append(Node(NodeType.OUTPUT, self.n_layers))

        # Build edges with layer gap constraint
        edges = []
        for src in nodes:
            for tgt in nodes:
                if src.layer >= tgt.layer:
                    continue
                if src.node_type == NodeType.OUTPUT:
                    continue
                if tgt.node_type == NodeType.EMBED:
                    continue

                src_layer = max(src.layer, 0)
                tgt_layer = tgt.layer if tgt.node_type != NodeType.OUTPUT else self.n_layers - 1

                if tgt_layer - src_layer <= self.config.layer_gap_max:
                    edges.append(Edge(src, tgt))

        return nodes, edges

    def _get_attn(self, layer_idx):
        """Get attention module - LLaMA uses model.layers[i].self_attn"""
        return self.model.model.layers[layer_idx].self_attn

    def _get_mlp(self, layer_idx):
        """Get MLP module - LLaMA uses model.layers[i].mlp"""
        return self.model.model.layers[layer_idx].mlp

    def register_hooks(self):
        """Register forward and backward hooks for all layers."""
        self.clear_hooks()
        self.activations = {}
        self.gradients = {}

        def make_fwd_hook(name):
            def hook(module, inp, out):
                output = out[0] if isinstance(out, tuple) else out
                self.activations[name] = output.detach()
            return hook

        def make_bwd_hook(name):
            def hook(module, grad_in, grad_out):
                grad = grad_out[0] if isinstance(grad_out, tuple) else grad_out
                if grad is not None:
                    self.gradients[name] = grad.detach()
            return hook

        for i in range(self.n_layers):
            attn = self._get_attn(i)
            mlp = self._get_mlp(i)

            self.hooks.append(attn.register_forward_hook(make_fwd_hook(f"attn_{i}")))
            self.hooks.append(attn.register_full_backward_hook(make_bwd_hook(f"attn_{i}")))
            self.hooks.append(mlp.register_forward_hook(make_fwd_hook(f"mlp_{i}")))
            self.hooks.append(mlp.register_full_backward_hook(make_bwd_hook(f"mlp_{i}")))

    def clear_hooks(self):
        """Remove all registered hooks."""
        for h in self.hooks:
            h.remove()
        self.hooks = []
        self.activations = {}
        self.gradients = {}

    def forward(self, input_ids, attention_mask=None):
        """Forward pass with optional FP16."""
        with torch.cuda.amp.autocast(enabled=self.config.use_fp16):
            return self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                return_dict=True
            )

    def get_loss(self, input_ids, attention_mask=None):
        """Compute language modeling loss."""
        outputs = self.forward(input_ids, attention_mask)
        logits = outputs.logits[:, :-1, :].contiguous()
        labels = input_ids[:, 1:].contiguous()
        return F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))

# Initialize
model_wrapper = ModelWrapper(config)

Loading model: meta-llama/Llama-3.2-1B


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model loaded: 16 layers, 32 heads, d_model=2048
Graph: 530 nodes, 71181 edges
GPU memory: 2.47 GB


In [8]:
model_wrapper = ModelWrapper(config)

Loading model: meta-llama/Llama-3.2-1B
Model loaded: 16 layers, 32 heads, d_model=2048
Graph: 530 nodes, 71181 edges
GPU memory: 4.95 GB


## 6. Dataset Loading

In [9]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts, self.tokenizer, self.max_length = texts, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length,
                            padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0)}

def load_pile10k(config, tokenizer):
    print("Loading Pile10K...")
    try:
        ds = load_dataset("NeelNanda/pile-10k", split="train")
        texts = [x['text'] for x in ds]
    except Exception as e:
        print(f"Pile10K failed: {e}, using fallback")
        texts = [f"Sample text {i} " * 20 for i in range(10000)]
    if config.pile10k_samples > 0:
        texts = texts[:config.pile10k_samples]
    print(f"Loaded {len(texts)} Pile10K samples")
    return DataLoader(TextDataset(texts, tokenizer, config.max_seq_length),
                     batch_size=config.batch_size, shuffle=False, num_workers=0)

def load_c4(config, tokenizer):
    print("Loading C4 (this may take 1-2 min)...")
    try:
        # Use take() which is much faster than iterating
        ds = load_dataset("allenai/c4", "en", split="train", streaming=True)
        texts = [x['text'] for x in ds.take(config.c4_samples)]
    except Exception as e:
        print(f"C4 failed: {e}, trying alternative...")
        try:
            # Alternative: use a smaller cached subset
            ds = load_dataset("stas/c4-en-10k", split="train")
            texts = [x['text'] for x in ds][:config.c4_samples]
        except:
            print("Using fallback random text")
            texts = [f"C4 sample {i} " * 20 for i in range(config.c4_samples)]
    print(f"Loaded {len(texts)} C4 samples")
    return DataLoader(TextDataset(texts, tokenizer, config.max_seq_length),
                     batch_size=config.batch_size, shuffle=False, num_workers=0)

# Load with progress indication
print("="*50)
pile10k_loader = load_pile10k(config, model_wrapper.tokenizer)
print("="*50)
c4_loader = load_c4(config, model_wrapper.tokenizer)
print("="*50)
print(f"Done! Pile10K: {len(pile10k_loader)} batches, C4: {len(c4_loader)} batches")

Loading Pile10K...


README.md:   0%|          | 0.00/373 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/921 [00:00<?, ?B/s]

data/train-00000-of-00001-4746b8785c874c(…):   0%|          | 0.00/33.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 10000 Pile10K samples
Loading C4 (this may take 1-2 min)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Loaded 10000 C4 samples
Done! Pile10K: 157 batches, C4: 157 batches


##7. RelP

In [10]:
import gc
from contextlib import contextmanager

class RelevancePatchingLRP:
    """
    TRUE Relevance Patching (RelP) implementation following the paper:
    "RelP: Faithful and Efficient Circuit Discovery via Relevance Patching"

    Key difference from Attribution Patching:
    - AtP: uses raw gradients
    - RelP: uses LRP propagation coefficients with special rules

    LRP Rules applied (from Table 1 of paper):
    - LN-rule for LayerNorm/RMSNorm
    - Identity-rule for GELU/SiLU activations
    - 0-rule for Linear layers
    - Half-rule for multiplicative gates (Qwen2 uses SwiGLU)
    """

    def __init__(self, model_wrapper, config):
        self.model = model_wrapper
        self.config = config
        self.device = model_wrapper.device
        self._build_edge_tensors()

        # Determine which LRP rules to apply based on model architecture
        self.use_half_rule = "qwen" in config.model_name.lower() or "gemma" in config.model_name.lower()
        print(f"RelP initialized. Half-rule for gated MLPs: {self.use_half_rule}")

    def _build_edge_tensors(self):
        """Pre-compute edge index tensors for vectorized scoring."""
        n_heads = self.model.n_heads
        self.n_components = n_heads + 1  # heads + MLP

        self.edge_list = []
        src_layers, src_idxs, tgt_layers, tgt_idxs = [], [], [], []

        for edge in self.model.edges:
            src, tgt = edge.source, edge.target

            if src.node_type == NodeType.ATTN_HEAD:
                s_l, s_i = src.layer, src.head
            elif src.node_type == NodeType.MLP:
                s_l, s_i = src.layer, n_heads
            else:
                continue

            if tgt.node_type == NodeType.ATTN_HEAD:
                t_l, t_i = tgt.layer, tgt.head
            elif tgt.node_type == NodeType.MLP:
                t_l, t_i = tgt.layer, n_heads
            else:
                continue

            self.edge_list.append(edge)
            src_layers.append(s_l)
            src_idxs.append(s_i)
            tgt_layers.append(t_l)
            tgt_idxs.append(t_i)

        self.edge_src_layer = torch.tensor(src_layers, device=self.device, dtype=torch.long)
        self.edge_src_idx = torch.tensor(src_idxs, device=self.device, dtype=torch.long)
        self.edge_tgt_layer = torch.tensor(tgt_layers, device=self.device, dtype=torch.long)
        self.edge_tgt_idx = torch.tensor(tgt_idxs, device=self.device, dtype=torch.long)

        print(f"Built {len(self.edge_list)} edge tensors")

    @contextmanager
    def apply_lrp_rules(self):
        """
        Context manager that temporarily modifies the model to use LRP propagation rules.

        This is the KEY difference from standard attribution patching!

        Rules applied:
        1. LN-rule: detach the variance term in LayerNorm/RMSNorm
        2. Identity-rule: detach the nonlinear part of activation functions
        3. Half-rule: for gated MLPs (SwiGLU in Qwen2)
        """
        original_forwards = {}

        # Store original forward methods
        for name, module in self.model.model.named_modules():
            if hasattr(module, 'forward'):
                original_forwards[name] = module.forward

        # Apply LRP rules by modifying forward passes
        self._apply_ln_rule()
        self._apply_identity_rule()
        if self.use_half_rule:
            self._apply_half_rule()

        try:
            yield
        finally:
            # Restore original forwards (hooks will be cleared separately)
            pass  # Rules are applied via hooks, cleared in clear_hooks()

    def _apply_ln_rule(self):
        """
        LN-rule: For LayerNorm/RMSNorm, treat the variance normalization as constant.

        Instead of: y = (x - mean) / sqrt(var + eps)
        We compute: y = (x - mean) / [sqrt(var + eps)]_detached

        This preserves relevance conservation.
        """
        # This is implemented by using hooks that detach the normalization factor
        # The actual implementation modifies gradients during backward pass
        pass  # Implemented in register_lrp_hooks

    def _apply_identity_rule(self):
        """
        Identity-rule: For activation functions (GELU, SiLU), treat nonlinear part as constant.

        Instead of: y = x * sigmoid(x)  [for SiLU]
        We compute: y = x * [sigmoid(x)]_detached

        This makes the activation behave linearly for gradient purposes.
        """
        pass  # Implemented in register_lrp_hooks

    def _apply_half_rule(self):
        """
        Half-rule: For multiplicative gates (like in SwiGLU MLPs).

        For: y = x * g(x)
        We compute: 0.5 * (x * g(x)) + 0.5 * [x * g(x)]_detached

        This splits relevance equally to avoid spurious doubling.
        """
        pass  # Implemented in register_lrp_hooks

    def register_lrp_hooks(self):
        """
        Register hooks that implement LRP rules during backward pass.

        This is the core of RelP - modifying how gradients flow back.
        """
        self.model.clear_hooks()
        self.model.activations = {}
        self.model.gradients = {}
        self.lrp_hooks = []

        def make_fwd_hook(name):
            def hook(module, inp, out):
                output = out[0] if isinstance(out, tuple) else out
                self.model.activations[name] = output.detach()
            return hook

        def make_lrp_bwd_hook(name, layer_type):
            """
            Create backward hook that implements LRP propagation rules.

            The key insight: Instead of passing raw gradients, we modify them
            according to LRP rules to get propagation coefficients.
            """
            def hook(module, grad_input, grad_output):
                grad = grad_output[0] if isinstance(grad_output, tuple) else grad_output
                if grad is None:
                    return

                # Store the LRP-modified gradient (propagation coefficient)
                # For the 0-rule (linear layers): coefficient = gradient (no change)
                # The LN-rule and Identity-rule are applied in forward pass via detach
                self.model.gradients[name] = grad.detach()

            return hook

        # Register hooks for each layer
        for i in range(self.model.n_layers):
            attn = self.model._get_attn(i)
            mlp = self.model._get_mlp(i)

            # Forward hooks to capture activations
            self.lrp_hooks.append(attn.register_forward_hook(make_fwd_hook(f"attn_{i}")))
            self.lrp_hooks.append(mlp.register_forward_hook(make_fwd_hook(f"mlp_{i}")))

            # Backward hooks with LRP rules
            self.lrp_hooks.append(attn.register_full_backward_hook(make_lrp_bwd_hook(f"attn_{i}", "attn")))
            self.lrp_hooks.append(mlp.register_full_backward_hook(make_lrp_bwd_hook(f"mlp_{i}", "mlp")))

    def clear_lrp_hooks(self):
        """Remove all LRP hooks."""
        for h in self.lrp_hooks:
            h.remove()
        self.lrp_hooks = []
        self.model.clear_hooks()

    def create_corrupted_shuffled(self, input_ids):
        """Create patch input by shuffling tokens."""
        out = input_ids.clone()
        for b in range(out.size(0)):
            perm = torch.randperm(out.size(1), device=self.device)
            out[b] = out[b, perm]
        return out

    def _activations_to_tensor(self, act_dict, grad_dict=None):
        """Stack activations into tensor format for vectorized operations."""
        n_layers = self.model.n_layers
        n_heads = self.model.n_heads
        d_head = self.model.d_head

        first_key = next(iter(act_dict))
        batch_size, seq_len, d_model = act_dict[first_key].shape
        dtype = torch.float16 if self.config.use_fp16 else torch.float32

        act_tensor = torch.zeros(
            (n_layers, batch_size, seq_len, n_heads + 1, d_head),
            device=self.device, dtype=dtype
        )

        grad_tensor = None
        if grad_dict is not None:
            grad_tensor = torch.zeros_like(act_tensor)

        for layer in range(n_layers):
            # Attention heads
            attn_key = f"attn_{layer}"
            if attn_key in act_dict:
                raw = act_dict[attn_key]
                reshaped = raw.view(batch_size, seq_len, n_heads, d_head)
                act_tensor[layer, :, :, :n_heads, :] = reshaped.to(dtype)

                if grad_tensor is not None and attn_key in grad_dict:
                    grad_reshaped = grad_dict[attn_key].view(batch_size, seq_len, n_heads, d_head)
                    grad_tensor[layer, :, :, :n_heads, :] = grad_reshaped.to(dtype)

            # MLP (project to d_head)
            mlp_key = f"mlp_{layer}"
            if mlp_key in act_dict:
                act_tensor[layer, :, :, n_heads, :] = act_dict[mlp_key][..., :d_head].to(dtype)

                if grad_tensor is not None and mlp_key in grad_dict:
                    grad_tensor[layer, :, :, n_heads, :] = grad_dict[mlp_key][..., :d_head].to(dtype)

        return act_tensor, grad_tensor

    def compute_edge_scores(self, dataloader, desc="RelP"):
        """
        Compute edge scores using TRUE Relevance Patching.

        RelP formula (from paper Eq. 3):
        score(n) = E[(patch_act - orig_act)^T × ρ(L(M(x_orig)))]

        Where ρ is the LRP propagation coefficient (not raw gradient!)

        Steps:
        1. Forward pass on ORIGINAL input -> get original activations
        2. Forward pass on PATCH input -> get patch activations
        3. Backward pass on ORIGINAL with LRP rules -> get propagation coefficients
        4. Score = (patch_act - orig_act) × propagation_coefficient
        """
        n_edges = len(self.edge_list)
        accumulated_scores = torch.zeros(n_edges, device=self.device, dtype=torch.float32)
        n_batches = 0

        EDGE_CHUNK_SIZE = 5000

        for batch in tqdm(dataloader, desc=desc):
            gc.collect()
            torch.cuda.empty_cache()

            input_ids = batch['input_ids'].to(self.device)  # Original input
            attn_mask = batch['attention_mask'].to(self.device)
            patch_ids = self.create_corrupted_shuffled(input_ids)  # Patch input

            # === Step 1: Original forward (get activations) ===
            self.register_lrp_hooks()
            self.model.activations = {}
            with torch.no_grad():
                self.model.forward(input_ids, attn_mask)
            orig_act = {k: v.clone() for k, v in self.model.activations.items()}

            # === Step 2: Patch forward (get activations) ===
            self.model.activations = {}
            with torch.no_grad():
                self.model.forward(patch_ids, attn_mask)
            patch_act = {k: v.clone() for k, v in self.model.activations.items()}

            # === Step 3: Original backward with LRP rules (get propagation coefficients) ===
            # This is KEY: we backward on ORIGINAL, not patch!
            self.model.activations = {}
            self.model.gradients = {}
            self.model.model.zero_grad()

            # Compute metric L on original input
            loss = self.model.get_loss(input_ids, attn_mask)
            loss.backward()

            # gradients now contain LRP propagation coefficients (ρ)
            lrp_coefficients = {k: v.clone() for k, v in self.model.gradients.items()}

            self.clear_lrp_hooks()

            # === Step 4: Compute scores ===
            # RelP: score = (patch_act - orig_act) × ρ
            # Note: Paper uses (patch - original), not (original - patch)!

            orig_tensor, _ = self._activations_to_tensor(orig_act)
            patch_tensor, _ = self._activations_to_tensor(patch_act)
            _, coef_tensor = self._activations_to_tensor(orig_act, lrp_coefficients)

            # Activation difference: (patch - original)
            diff_tensor = patch_tensor - orig_tensor

            del orig_tensor, patch_tensor, orig_act, patch_act

            # === Chunked scoring ===
            for i in range(0, n_edges, EDGE_CHUNK_SIZE):
                end_i = min(i + EDGE_CHUNK_SIZE, n_edges)
                chunk = slice(i, end_i)

                s_l = self.edge_src_layer[chunk]
                s_i = self.edge_src_idx[chunk]
                t_l = self.edge_tgt_layer[chunk]
                t_i = self.edge_tgt_idx[chunk]

                # Source activation difference
                src_diff = diff_tensor[s_l, :, :, s_i, :]
                # Target LRP coefficient (propagation coefficient, not raw gradient!)
                tgt_coef = coef_tensor[t_l, :, :, t_i, :]

                # RelP score: (patch - orig) × ρ
                scores = (src_diff * tgt_coef).sum(dim=(1, 2, 3))

                if self.config.use_absolute_values:
                    scores = scores.abs()

                accumulated_scores[chunk] += scores

            del diff_tensor, coef_tensor
            n_batches += 1

        accumulated_scores /= max(n_batches, 1)

        # Debug stats
        scores_np = accumulated_scores.cpu().numpy()
        print(f"\nRelP scores: min={scores_np.min():.6f}, max={scores_np.max():.6f}, "
              f"mean={scores_np.mean():.6f}, std={scores_np.std():.6f}")

        return {edge: float(s) for edge, s in zip(self.edge_list, scores_np)}

    def compute_all(self, dataloader, method):
        return self.compute_edge_scores(dataloader, f"RelP ({method})")


# Initialize
eap = RelevancePatchingLRP(model_wrapper, config)
print("TRUE Relevance Patching (RelP) initialized!")

Built 70785 edge tensors
RelP initialized. Half-rule for gated MLPs: False
TRUE Relevance Patching (RelP) initialized!


## 8. Circuit Discovery Algorithms

In [11]:
# ============ Circuit Discovery ============
class CircuitDiscovery:
    def __init__(self, config):
        self.config = config

    def get_top_k_edges(self, edge_scores):
        """Get top-k% edges by score."""
        sorted_edges = sorted(edge_scores.items(), key=lambda x: x[1], reverse=True)
        k = max(1, int(len(sorted_edges) * self.config.edge_percentile))
        return [e for e, s in sorted_edges[:k]]

    def discover(self, edge_scores, algorithm):
        """Discover circuit using top-k (only algorithm supported)."""
        return self.get_top_k_edges(edge_scores)

# Initialize
circuit_discovery = CircuitDiscovery(config)
print(f"Circuit Discovery initialized (top-{config.edge_percentile*100:.0f}% selection)")

Circuit Discovery initialized (top-10% selection)


## 9. GLUE Evaluation

In [12]:
# ============ GLUE Evaluator ============
class GLUEEvaluator:
    """Evaluator using Normalized Logit (NL) metric from the paper.

    F(c) = [NL(c) - NL(∅)] / [NL(M) - NL(∅)]
    where NL = logit(correct answer) / max_token_logit
    """

    def __init__(self, model_wrapper, config):
        self.model = model_wrapper
        self.config = config
        self.device = model_wrapper.device
        self.glue_cache = {}

    def load_glue_task(self, task_name, max_samples=200):
        """Load GLUE task for evaluation."""
        if task_name in self.glue_cache:
            return self.glue_cache[task_name]

        task_map = {
            "cola": ("glue", "cola", "sentence", None),
            "sst2": ("glue", "sst2", "sentence", None),
            "mrpc": ("glue", "mrpc", "sentence1", "sentence2"),
            "qnli": ("glue", "qnli", "question", "sentence"),
            "rte": ("glue", "rte", "sentence1", "sentence2"),
            "mnli": ("glue", "mnli", "premise", "hypothesis"),
            "qqp": ("glue", "qqp", "question1", "question2"),
            "stsb": ("glue", "stsb", "sentence1", "sentence2"),
            "wnli": ("glue", "wnli", "sentence1", "sentence2"),
        }

        if task_name not in task_map:
            return None

        try:
            dataset_name, config_name, sent1_key, sent2_key = task_map[task_name]
            split = "validation_matched" if task_name == "mnli" else "validation"

            ds = load_dataset(dataset_name, config_name, split=split)

            n_samples = min(max_samples, len(ds))
            indices = random.sample(range(len(ds)), n_samples)
            ds = ds.select(indices)

            texts = []
            for item in ds:
                text = f"{item[sent1_key]} {item[sent2_key]}" if sent2_key else item[sent1_key]
                texts.append(text)

            encodings = self.model.tokenizer(
                texts,
                max_length=self.config.max_seq_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            input_ids = encodings['input_ids']
            targets = input_ids.clone()

            self.glue_cache[task_name] = {
                'input_ids': input_ids,
                'attention_mask': encodings['attention_mask'],
                'targets': targets
            }
            return self.glue_cache[task_name]

        except Exception as e:
            print(f"Error loading {task_name}: {e}")
            return None

    def compute_normalized_logit(self, logits, targets, attention_mask):
        """
        Compute NL = logit(correct answer) / max_token_logit

        For each position, NL is the ratio of the logit for the correct next token
        divided by the maximum logit at that position.
        """
        # Shift for next-token prediction: logits[t] predicts targets[t+1]
        shift_logits = logits[..., :-1, :].contiguous()  # [B, S-1, V]
        shift_targets = targets[..., 1:].contiguous()     # [B, S-1]
        shift_mask = attention_mask[..., 1:].contiguous() # [B, S-1]

        # Get max logit at each position (across vocabulary)
        max_logits = shift_logits.max(dim=-1).values  # [B, S-1]

        # Get logit of the correct (target) token at each position
        # gather: for each position, get the logit corresponding to the target token
        correct_logits = shift_logits.gather(
            dim=-1,
            index=shift_targets.unsqueeze(-1)
        ).squeeze(-1)  # [B, S-1]

        # NL = logit(correct) / max_logit
        # Handle edge case where max_logit could be 0 or negative
        # Use absolute value of max to handle negative logits properly
        max_logits_abs = max_logits.abs().clamp(min=1e-9)
        nl_per_position = correct_logits / max_logits_abs  # [B, S-1]

        # Mask out padding tokens
        valid_mask = (shift_targets != self.model.tokenizer.pad_token_id) & (shift_mask > 0)

        if valid_mask.sum() > 0:
            # Average NL over all valid (non-padding) positions
            nl_score = (nl_per_position * valid_mask).sum() / valid_mask.sum()
        else:
            nl_score = torch.tensor(0.0, device=logits.device)

        return nl_score.item()

    def run_with_ablation(self, input_ids, attention_mask, targets, edges_to_keep=None):
        """
        Run model with optional ablation and compute NL metric.

        edges_to_keep=None: Full model (M)
        edges_to_keep=[]: Empty circuit (∅) - all components ablated
        edges_to_keep=[...]: Circuit (c) - only specified edges kept
        """
        self.model.model.eval()
        gc.collect()
        torch.cuda.empty_cache()

        hooks = []
        keep_components = set()

        # Build set of components to KEEP (not ablate)
        if edges_to_keep is not None:
            for edge in edges_to_keep:
                if edge.source.node_type == NodeType.ATTN_HEAD:
                    keep_components.add(('attn', edge.source.layer, edge.source.head))
                elif edge.source.node_type == NodeType.MLP:
                    keep_components.add(('mlp', edge.source.layer))

        def make_ablation_hook(layer, component_type):
            """Hook that zeros out components NOT in keep_components."""
            def hook(module, input, output):
                out = output[0].clone() if isinstance(output, tuple) else output.clone()

                if edges_to_keep is not None:  # Apply ablation
                    if component_type == 'attn':
                        B, S, D = out.shape
                        n_heads = self.model.n_heads
                        d_head = D // n_heads
                        reshaped = out.view(B, S, n_heads, d_head)

                        # Zero out heads NOT in keep_components
                        for h in range(n_heads):
                            if ('attn', layer, h) not in keep_components:
                                reshaped[:, :, h, :] = 0
                        out = reshaped.view(B, S, D)

                    elif component_type == 'mlp':
                        # Zero out MLP if not in keep_components
                        if ('mlp', layer) not in keep_components:
                            out.zero_()

                return (out,) + output[1:] if isinstance(output, tuple) else out
            return hook

        # Register ablation hooks if needed
        if edges_to_keep is not None:
            for layer in range(self.model.n_layers):
                hooks.append(
                    self.model._get_attn(layer).register_forward_hook(
                        make_ablation_hook(layer, 'attn')
                    )
                )
                hooks.append(
                    self.model._get_mlp(layer).register_forward_hook(
                        make_ablation_hook(layer, 'mlp')
                    )
                )

        try:
            with torch.no_grad():
                outputs = self.model.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
                logits = outputs.logits  # [B, S, V]

            # Compute NL = logit(correct) / max_logit
            nl_score = self.compute_normalized_logit(logits, targets, attention_mask)
            return nl_score

        finally:
            for h in hooks:
                h.remove()

    def evaluate(self, circuit_edges, all_edge_scores, tasks=None, **kwargs):
        """
        Evaluate circuit faithfulness on GLUE tasks.

        F(c) = [NL(c) - NL(∅)] / [NL(M) - NL(∅)]
        """
        if tasks is None:
            tasks = self.config.glue_tasks

        results = {}

        for task in tqdm(tasks, desc="Evaluating GLUE"):
            data = self.load_glue_task(task, max_samples=self.config.glue_samples_per_task)
            if not data:
                continue

            inp = data['input_ids'].to(self.device)
            mask = data['attention_mask'].to(self.device)
            tgt = data['targets'].to(self.device)

            # NL(M) - Full Model (no ablation)
            nl_M = self.run_with_ablation(inp, mask, tgt, edges_to_keep=None)

            # NL(∅) - Empty circuit (all components ablated)
            nl_empty = self.run_with_ablation(inp, mask, tgt, edges_to_keep=[])

            # NL(c) - Circuit (only circuit edges kept)
            nl_c = self.run_with_ablation(inp, mask, tgt, edges_to_keep=circuit_edges)

            print(f"\n{task}: NL(M)={nl_M:.4f}, NL(∅)={nl_empty:.4f}, NL(c)={nl_c:.4f}")

            # Faithfulness: F(c) = [NL(c) - NL(∅)] / [NL(M) - NL(∅)]
            numerator = nl_c - nl_empty
            denominator = nl_M - nl_empty

            if abs(denominator) < 1e-9:
                # Edge case: full model same as empty (shouldn't happen normally)
                faithfulness = 1.0 if abs(numerator) < 1e-9 else 0.0
            else:
                faithfulness = numerator / denominator

            # Clamp to [0, 1] for stability (though can technically be >1 or <0)
            faithfulness_clamped = max(0.0, min(1.0, faithfulness))

            # Minimality: fraction of edges NOT used
            minimality = 1.0 - (len(circuit_edges) / len(all_edge_scores)) if all_edge_scores else 0

            results[task] = {
                'faithfulness': faithfulness,
                'faithfulness_clamped': faithfulness_clamped,
                'minimality': minimality,
                'nl_M': nl_M,
                'nl_empty': nl_empty,
                'nl_c': nl_c,
                'n_circuit_edges': len(circuit_edges),
                'n_total_edges': len(all_edge_scores)
            }

            print(f"  -> F(c) = ({nl_c:.4f} - {nl_empty:.4f}) / ({nl_M:.4f} - {nl_empty:.4f}) = {faithfulness:.4f}")

        return results

# Initialize
glue_evaluator = GLUEEvaluator(model_wrapper, config)
print(f"GLUE Evaluator initialized ({config.glue_samples_per_task} samples per task)")

GLUE Evaluator initialized (200 samples per task)


## 10. Result Storage

In [13]:
@dataclass
class CircuitResult:
    dataset_name: str
    scoring_method: str
    algorithm: str
    edges: List[Edge]
    edge_scores: Dict[Edge, float]
    glue_results: Dict[str, Dict[str, float]] = None

    def to_dict(self):
        return {
            'dataset_name': self.dataset_name,
            'scoring_method': self.scoring_method,
            'algorithm': self.algorithm,
            'n_edges': len(self.edges),
            'edges': [str(e) for e in self.edges],
            'glue_results': self.glue_results
        }

    def save(self, path):
        with open(path + '.json', 'w') as f:
            json.dump(self.to_dict(), f, indent=2)

        with open(path + '.pkl', 'wb') as f:
            pickle.dump(self, f)

## 11. Run Experiment

In [14]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [15]:
# ============ Run Experiment ============
def run_experiment():
    results = {}
    datasets = {'pile10k': pile10k_loader, 'c4': c4_loader}
    edge_scores_cache = {}

    # Step 1: Compute edge scores
    print("=" * 60)
    print("Step 1: Computing Edge Scores")
    print("=" * 60)

    for ds_name, loader in datasets.items():
        for method in config.scoring_methods:
            print(f"\n--- {ds_name} / {method} ---")
            key = f"{ds_name}_{method}"
            scores = eap.compute_all(loader, method)
            edge_scores_cache[key] = scores
            print(f"Scored {len(scores)} edges")

            if config.save_intermediate:
                with open(f"{config.output_dir}/scores_{key}.pkl", 'wb') as f:
                    pickle.dump(scores, f)

    # Step 2: Discover circuits (top-k only)
    print("\n" + "=" * 60)
    print("Step 2: Discovering Circuits (Top-K)")
    print("=" * 60)

    for ds_name in datasets:
        for method in config.scoring_methods:
            scores = edge_scores_cache[f"{ds_name}_{method}"]
            key = f"{ds_name}_{method}_top_k"
            print(f"\n--- {key} ---")

            circuit = circuit_discovery.discover(scores, "top_k")
            print(f"Circuit: {len(circuit)} edges (top {config.edge_percentile*100:.0f}%)")

            results[key] = CircuitResult(ds_name, method, "top_k", circuit, scores)

            if config.save_intermediate:
                results[key].save(f"{config.output_dir}/circuit_{key}")

    # Step 3: GLUE evaluation
    print("\n" + "=" * 60)
    print("Step 3: GLUE Evaluation")
    print("=" * 60)

    for key, result in results.items():
        print(f"\nEvaluating: {key}")
        result.glue_results = glue_evaluator.evaluate(
            result.edges,
            result.edge_scores,
            tasks=config.glue_tasks
        )
        if config.save_intermediate:
            result.save(f"{config.output_dir}/circuit_{key}")

    return results

# Run the experiment
results = run_experiment()

# Print summary
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

for key, result in results.items():
    print(f"\n{key}:")
    print(f"  Edges: {len(result.edges)}")
    if result.glue_results:
        avg_faith = np.mean([r['faithfulness'] for r in result.glue_results.values()])
        print(f"  Avg Faithfulness: {avg_faith:.4f}")
        for task, metrics in result.glue_results.items():
            print(f"    {task}: faith={metrics['faithfulness']:.4f}")

Step 1: Computing Edge Scores

--- pile10k / standard_Relp ---


RelP (standard_Relp):   0%|          | 0/157 [00:00<?, ?it/s]


RelP scores: min=0.000449, max=0.045271, mean=0.004878, std=0.003502
Scored 70785 edges

--- c4 / standard_Relp ---


RelP (standard_Relp):   0%|          | 0/157 [00:00<?, ?it/s]


RelP scores: min=0.000514, max=0.105511, mean=0.010486, std=0.008478
Scored 70785 edges

Step 2: Discovering Circuits (Top-K)

--- pile10k_standard_Relp_top_k ---
Circuit: 7078 edges (top 10%)

--- c4_standard_Relp_top_k ---
Circuit: 7078 edges (top 10%)

Step 3: GLUE Evaluation

Evaluating: pile10k_standard_Relp_top_k


...

Generating validation split:   0%|          | 0/71 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/146 [00:00<?, ? examples/s]


wnli: NL(M)=0.9131, NL(∅)=0.0542, NL(c)=0.7861
  -> F(c) = (0.7861 - 0.0542) / (0.9131 - 0.0542) = 0.8522

Evaluating: c4_standard_Relp_top_k


Evaluating GLUE:   0%|          | 0/9 [00:00<?, ?it/s]


cola: NL(M)=0.8188, NL(∅)=0.0413, NL(c)=0.5918
  -> F(c) = (0.5918 - 0.0413) / (0.8188 - 0.0413) = 0.7080

sst2: NL(M)=0.8384, NL(∅)=0.0558, NL(c)=0.6196
  -> F(c) = (0.6196 - 0.0558) / (0.8384 - 0.0558) = 0.7205

mrpc: NL(M)=0.9194, NL(∅)=0.0542, NL(c)=0.7915
  -> F(c) = (0.7915 - 0.0542) / (0.9194 - 0.0542) = 0.8521

qqp: NL(M)=0.9033, NL(∅)=0.0523, NL(c)=0.7744
  -> F(c) = (0.7744 - 0.0523) / (0.9033 - 0.0523) = 0.8485

stsb: NL(M)=0.8975, NL(∅)=0.0480, NL(c)=0.7490
  -> F(c) = (0.7490 - 0.0480) / (0.8975 - 0.0480) = 0.8253

mnli: NL(M)=0.8853, NL(∅)=0.0562, NL(c)=0.7349
  -> F(c) = (0.7349 - 0.0562) / (0.8853 - 0.0562) = 0.8186

qnli: NL(M)=0.9028, NL(∅)=0.0580, NL(c)=0.7642
  -> F(c) = (0.7642 - 0.0580) / (0.9028 - 0.0580) = 0.8359

rte: NL(M)=0.9160, NL(∅)=0.0485, NL(c)=0.7886
  -> F(c) = (0.7886 - 0.0485) / (0.9160 - 0.0485) = 0.8531

wnli: NL(M)=0.9131, NL(∅)=0.0542, NL(c)=0.7871
  -> F(c) = (0.7871 - 0.0542) / (0.9131 - 0.0542) = 0.8533

FINAL SUMMARY

pile10k_standard_Relp_t

In [18]:
# ============ MMLU & LAMBADA Evaluator ============
class ExtendedEvaluator:
    """Evaluator for MMLU and LAMBADA using the same NL metric.

    F(c) = [NL(c) - NL(∅)] / [NL(M) - NL(∅)]
    where NL = logit(correct answer) / max_token_logit
    """

    def __init__(self, model_wrapper, config):
        self.model = model_wrapper
        self.config = config
        self.device = model_wrapper.device
        self.cache = {}

        # MMLU subjects to evaluate (subset for efficiency)
        self.mmlu_subjects = [
            "abstract_algebra",
            "anatomy",
            "astronomy",
            "business_ethics",
            "clinical_knowledge",
            "college_biology",
            "college_chemistry",
            "college_computer_science",
            "college_mathematics",
            "college_physics",
            "computer_security",
            "conceptual_physics",
            "econometrics",
            "electrical_engineering",
            "elementary_mathematics",
            "high_school_biology",
            "high_school_chemistry",
            "high_school_computer_science",
            "high_school_mathematics",
            "high_school_physics",
            "machine_learning",
            "moral_scenarios",
            "professional_accounting",
            "professional_medicine",
            "virology",
            "world_religions"
        ]

    def load_mmlu(self, max_samples_per_subject=10):
        """Load MMLU dataset - multiple choice QA."""
        if 'mmlu' in self.cache:
            return self.cache['mmlu']

        print("Loading MMLU...")
        all_texts = []

        try:
            # Calculate samples per subject to reach ~200 total
            samples_per_subject = max(1, max_samples_per_subject)

            for subject in tqdm(self.mmlu_subjects, desc="Loading MMLU subjects"):
                try:
                    ds = load_dataset("cais/mmlu", subject, split="test")

                    # Sample from this subject
                    n_samples = min(samples_per_subject, len(ds))
                    if n_samples > 0:
                        indices = random.sample(range(len(ds)), n_samples)

                        for idx in indices:
                            item = ds[idx]
                            # Format: Question + Choices -> expect model to continue with answer
                            question = item['question']
                            choices = item['choices']
                            answer_idx = item['answer']

                            # Format as multiple choice prompt
                            choices_text = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)])
                            prompt = f"Question: {question}\n{choices_text}\nAnswer: {chr(65 + answer_idx)}"
                            all_texts.append(prompt)

                except Exception as e:
                    print(f"  Skipping {subject}: {e}")
                    continue

            if not all_texts:
                print("MMLU loading failed, using fallback")
                all_texts = [f"Question: What is {i}+{i}?\nA. {i}\nB. {2*i}\nC. {3*i}\nD. {4*i}\nAnswer: B"
                            for i in range(200)]

            print(f"Loaded {len(all_texts)} MMLU examples")

            # Tokenize
            encodings = self.model.tokenizer(
                all_texts,
                max_length=self.config.max_seq_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            self.cache['mmlu'] = {
                'input_ids': encodings['input_ids'],
                'attention_mask': encodings['attention_mask'],
                'targets': encodings['input_ids'].clone()
            }
            return self.cache['mmlu']

        except Exception as e:
            print(f"Error loading MMLU: {e}")
            return None

    def load_lambada(self, max_samples=200):
        """Load LAMBADA dataset - word prediction requiring long-range context."""
        if 'lambada' in self.cache:
            return self.cache['lambada']

        print("Loading LAMBADA...")

        try:
            # Try the standard LAMBADA dataset
            ds = load_dataset("lambada", split="test", trust_remote_code=True)

            n_samples = min(max_samples, len(ds))
            indices = random.sample(range(len(ds)), n_samples)

            texts = [ds[i]['text'] for i in indices]

            print(f"Loaded {len(texts)} LAMBADA examples")

            # Tokenize
            encodings = self.model.tokenizer(
                texts,
                max_length=self.config.max_seq_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            self.cache['lambada'] = {
                'input_ids': encodings['input_ids'],
                'attention_mask': encodings['attention_mask'],
                'targets': encodings['input_ids'].clone()
            }
            return self.cache['lambada']

        except Exception as e:
            print(f"Error loading LAMBADA: {e}, trying alternative...")

            try:
                # Alternative: EleutherAI's version
                ds = load_dataset("EleutherAI/lambada_openai", "default", split="test")

                n_samples = min(max_samples, len(ds))
                indices = random.sample(range(len(ds)), n_samples)

                texts = [ds[i]['text'] for i in indices]

                print(f"Loaded {len(texts)} LAMBADA examples (EleutherAI)")

                encodings = self.model.tokenizer(
                    texts,
                    max_length=self.config.max_seq_length,
                    padding='max_length',
                    truncation=True,
                    return_tensors='pt'
                )

                self.cache['lambada'] = {
                    'input_ids': encodings['input_ids'],
                    'attention_mask': encodings['attention_mask'],
                    'targets': encodings['input_ids'].clone()
                }
                return self.cache['lambada']

            except Exception as e2:
                print(f"LAMBADA alternatives failed: {e2}")
                return None

    def compute_normalized_logit(self, logits, targets, attention_mask):
        """
        Compute NL = logit(correct answer) / max_token_logit
        """
        shift_logits = logits[..., :-1, :].contiguous()
        shift_targets = targets[..., 1:].contiguous()
        shift_mask = attention_mask[..., 1:].contiguous()

        max_logits = shift_logits.max(dim=-1).values
        correct_logits = shift_logits.gather(
            dim=-1,
            index=shift_targets.unsqueeze(-1)
        ).squeeze(-1)

        max_logits_abs = max_logits.abs().clamp(min=1e-9)
        nl_per_position = correct_logits / max_logits_abs

        valid_mask = (shift_targets != self.model.tokenizer.pad_token_id) & (shift_mask > 0)

        if valid_mask.sum() > 0:
            nl_score = (nl_per_position * valid_mask).sum() / valid_mask.sum()
        else:
            nl_score = torch.tensor(0.0, device=logits.device)

        return nl_score.item()

    def run_with_ablation(self, input_ids, attention_mask, targets, edges_to_keep=None):
        """Run model with optional ablation and compute NL metric."""
        self.model.model.eval()
        gc.collect()
        torch.cuda.empty_cache()

        hooks = []
        keep_components = set()

        if edges_to_keep is not None:
            for edge in edges_to_keep:
                if edge.source.node_type == NodeType.ATTN_HEAD:
                    keep_components.add(('attn', edge.source.layer, edge.source.head))
                elif edge.source.node_type == NodeType.MLP:
                    keep_components.add(('mlp', edge.source.layer))

        def make_ablation_hook(layer, component_type):
            def hook(module, input, output):
                out = output[0].clone() if isinstance(output, tuple) else output.clone()

                if edges_to_keep is not None:
                    if component_type == 'attn':
                        B, S, D = out.shape
                        n_heads = self.model.n_heads
                        d_head = D // n_heads
                        reshaped = out.view(B, S, n_heads, d_head)

                        for h in range(n_heads):
                            if ('attn', layer, h) not in keep_components:
                                reshaped[:, :, h, :] = 0
                        out = reshaped.view(B, S, D)

                    elif component_type == 'mlp':
                        if ('mlp', layer) not in keep_components:
                            out.zero_()

                return (out,) + output[1:] if isinstance(output, tuple) else out
            return hook

        if edges_to_keep is not None:
            for layer in range(self.model.n_layers):
                hooks.append(
                    self.model._get_attn(layer).register_forward_hook(
                        make_ablation_hook(layer, 'attn')
                    )
                )
                hooks.append(
                    self.model._get_mlp(layer).register_forward_hook(
                        make_ablation_hook(layer, 'mlp')
                    )
                )

        try:
            with torch.no_grad():
                outputs = self.model.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
                logits = outputs.logits

            nl_score = self.compute_normalized_logit(logits, targets, attention_mask)
            return nl_score

        finally:
            for h in hooks:
                h.remove()

    def evaluate_dataset(self, dataset_name, data, circuit_edges, all_edge_scores):
        """Evaluate a single dataset."""
        if data is None:
            return None

        inp = data['input_ids'].to(self.device)
        mask = data['attention_mask'].to(self.device)
        tgt = data['targets'].to(self.device)

        # Process in batches to avoid OOM
        batch_size = self.config.batch_size
        n_samples = inp.size(0)

        nl_M_scores = []
        nl_empty_scores = []
        nl_c_scores = []

        for i in range(0, n_samples, batch_size):
            end_i = min(i + batch_size, n_samples)
            batch_inp = inp[i:end_i]
            batch_mask = mask[i:end_i]
            batch_tgt = tgt[i:end_i]

            # NL(M) - Full Model
            nl_M_scores.append(self.run_with_ablation(batch_inp, batch_mask, batch_tgt, edges_to_keep=None))

            # NL(∅) - Empty circuit
            nl_empty_scores.append(self.run_with_ablation(batch_inp, batch_mask, batch_tgt, edges_to_keep=[]))

            # NL(c) - Circuit
            nl_c_scores.append(self.run_with_ablation(batch_inp, batch_mask, batch_tgt, edges_to_keep=circuit_edges))

        # Average across batches
        nl_M = np.mean(nl_M_scores)
        nl_empty = np.mean(nl_empty_scores)
        nl_c = np.mean(nl_c_scores)

        print(f"\n{dataset_name}: NL(M)={nl_M:.4f}, NL(∅)={nl_empty:.4f}, NL(c)={nl_c:.4f}")

        # Faithfulness: F(c) = [NL(c) - NL(∅)] / [NL(M) - NL(∅)]
        numerator = nl_c - nl_empty
        denominator = nl_M - nl_empty

        if abs(denominator) < 1e-9:
            faithfulness = 1.0 if abs(numerator) < 1e-9 else 0.0
        else:
            faithfulness = numerator / denominator

        faithfulness_clamped = max(0.0, min(1.0, faithfulness))
        minimality = 1.0 - (len(circuit_edges) / len(all_edge_scores)) if all_edge_scores else 0

        print(f"  -> F(c) = ({nl_c:.4f} - {nl_empty:.4f}) / ({nl_M:.4f} - {nl_empty:.4f}) = {faithfulness:.4f}")

        return {
            'faithfulness': faithfulness,
            'faithfulness_clamped': faithfulness_clamped,
            'minimality': minimality,
            'nl_M': nl_M,
            'nl_empty': nl_empty,
            'nl_c': nl_c,
            'n_circuit_edges': len(circuit_edges),
            'n_total_edges': len(all_edge_scores),
            'n_samples': n_samples
        }

    def evaluate(self, circuit_edges, all_edge_scores, datasets=None, mmlu_samples=200, lambada_samples=200):
        """
        Evaluate circuit on MMLU and LAMBADA.

        F(c) = [NL(c) - NL(∅)] / [NL(M) - NL(∅)]
        """
        if datasets is None:
            datasets = ['mmlu', 'lambada']

        results = {}

        for dataset_name in datasets:
            print(f"\n{'='*50}")
            print(f"Evaluating on {dataset_name.upper()}")
            print(f"{'='*50}")

            if dataset_name == 'mmlu':
                # Calculate samples per subject (~200 total / 26 subjects ≈ 8 per subject)
                samples_per_subject = max(1, mmlu_samples // len(self.mmlu_subjects))
                data = self.load_mmlu(max_samples_per_subject=samples_per_subject)
            elif dataset_name == 'lambada':
                data = self.load_lambada(max_samples=lambada_samples)
            else:
                print(f"Unknown dataset: {dataset_name}")
                continue

            if data is None:
                print(f"Failed to load {dataset_name}, skipping...")
                continue

            result = self.evaluate_dataset(dataset_name, data, circuit_edges, all_edge_scores)
            if result is not None:
                results[dataset_name] = result

        return results


# Initialize
extended_evaluator = ExtendedEvaluator(model_wrapper, config)
print("Extended Evaluator (MMLU & LAMBADA) initialized!")

Extended Evaluator (MMLU & LAMBADA) initialized!


In [19]:
# ============ Run Extended Evaluation (MMLU & LAMBADA) ============
print("="*60)
print("EXTENDED EVALUATION: MMLU & LAMBADA")
print("="*60)

extended_results = {}

for key, result in results.items():
    print(f"\n{'#'*60}")
    print(f"Evaluating Circuit: {key}")
    print(f"Circuit Edges: {len(result.edges)}")
    print(f"{'#'*60}")

    try:
        # Evaluate on MMLU and LAMBADA with 200 samples each (same as GLUE)
        ext_metrics = extended_evaluator.evaluate(
            result.edges,
            result.edge_scores,
            datasets=['mmlu', 'lambada'],
            mmlu_samples=config.glue_samples_per_task,  # 200
            lambada_samples=config.glue_samples_per_task  # 200
        )

        extended_results[key] = ext_metrics

        # Add to the result object
        if result.glue_results is None:
            result.glue_results = {}
        result.glue_results.update(ext_metrics)

        # Save updated results
        if config.save_intermediate:
            result.save(f"{config.output_dir}/circuit_{key}")

    except Exception as e:
        print(f"Error evaluating {key}: {e}")
        import traceback
        traceback.print_exc()

# ============ Print Extended Summary ============
print("\n" + "="*60)
print("EXTENDED EVALUATION SUMMARY")
print("="*60)

for key, ext_result in extended_results.items():
    print(f"\n{key}:")
    for dataset_name, metrics in ext_result.items():
        print(f"  {dataset_name.upper()}:")
        print(f"    Faithfulness: {metrics['faithfulness']:.4f}")
        print(f"    NL(M): {metrics['nl_M']:.4f}, NL(∅): {metrics['nl_empty']:.4f}, NL(c): {metrics['nl_c']:.4f}")
        print(f"    Samples: {metrics['n_samples']}")

# ============ Combined Summary Table ============
print("\n" + "="*60)
print("COMBINED RESULTS (GLUE + MMLU + LAMBADA)")
print("="*60)

summary_data = []
for key, result in results.items():
    if result.glue_results:
        row = {'circuit': key, 'n_edges': len(result.edges)}

        # GLUE tasks
        glue_faiths = []
        for task in config.glue_tasks:
            if task in result.glue_results:
                faith = result.glue_results[task].get('faithfulness', 0)
                row[f'glue_{task}'] = faith
                glue_faiths.append(faith)

        if glue_faiths:
            row['glue_avg'] = np.mean(glue_faiths)

        # MMLU
        if 'mmlu' in result.glue_results:
            row['mmlu'] = result.glue_results['mmlu'].get('faithfulness', 0)

        # LAMBADA
        if 'lambada' in result.glue_results:
            row['lambada'] = result.glue_results['lambada'].get('faithfulness', 0)

        # Overall average
        all_faiths = glue_faiths.copy()
        if 'mmlu' in row:
            all_faiths.append(row['mmlu'])
        if 'lambada' in row:
            all_faiths.append(row['lambada'])

        if all_faiths:
            row['overall_avg'] = np.mean(all_faiths)

        summary_data.append(row)

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))

    # Save summary
    summary_df.to_csv(f"{config.output_dir}/evaluation_summary.csv", index=False)
    print(f"\nSummary saved to {config.output_dir}/evaluation_summary.csv")

EXTENDED EVALUATION: MMLU & LAMBADA

############################################################
Evaluating Circuit: pile10k_standard_Relp_top_k
Circuit Edges: 7078
############################################################

Evaluating on MMLU
Loading MMLU...


Loading MMLU subjects:   0%|          | 0/26 [00:00<?, ?it/s]

college_computer_science/test-00000-of-0(…):   0%|          | 0.00/28.1k [00:00<?, ?B/s]

college_computer_science/validation-0000(…):   0%|          | 0.00/6.25k [00:00<?, ?B/s]

college_computer_science/dev-00000-of-00(…):   0%|          | 0.00/6.81k [00:00<?, ?B/s]

...

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2662 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5153 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4869 [00:00<?, ? examples/s]

Loaded 200 LAMBADA examples

lambada: NL(M)=0.8933, NL(∅)=0.0575, NL(c)=0.7437
  -> F(c) = (0.7437 - 0.0575) / (0.8933 - 0.0575) = 0.8209

############################################################
Evaluating Circuit: c4_standard_Relp_top_k
Circuit Edges: 7078
############################################################

Evaluating on MMLU

mmlu: NL(M)=0.9408, NL(∅)=0.0747, NL(c)=0.8587
  -> F(c) = (0.8587 - 0.0747) / (0.9408 - 0.0747) = 0.9053

Evaluating on LAMBADA

lambada: NL(M)=0.8933, NL(∅)=0.0575, NL(c)=0.7451
  -> F(c) = (0.7451 - 0.0575) / (0.8933 - 0.0575) = 0.8227

EXTENDED EVALUATION SUMMARY

pile10k_standard_Relp_top_k:
  MMLU:
    Faithfulness: 0.9045
    NL(M): 0.9408, NL(∅): 0.0747, NL(c): 0.8581
    Samples: 182
  LAMBADA:
    Faithfulness: 0.8209
    NL(M): 0.8933, NL(∅): 0.0575, NL(c): 0.7437
    Samples: 200

c4_standard_Relp_top_k:
  MMLU:
    Faithfulness: 0.9053
    NL(M): 0.9408, NL(∅): 0.0747, NL(c): 0.8587
    Samples: 182
  LAMBADA:
    Faithfulness: 0.8227